<a href="https://colab.research.google.com/github/roopaam/MB-Proxy/blob/main/PEP_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from scipy.stats import entropy
from scipy.special import rel_entr
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


def load_and_preprocess_adult() -> tuple[pd.DataFrame, str, str, str]:
    """
    Load and preprocess UCI Adult dataset.

    Returns:
        df: Preprocessed DataFrame
        target: Target column name (income)
        sensitive: Sensitive attribute (sex)
        proxy: Proxy attribute (relationship)
    """
    columns = [
        "age", "workclass", "fnlwgt", "education", "education-num", "marital-status",
        "occupation", "relationship", "race", "sex", "capital-gain", "capital-loss",
        "hours-per-week", "native-country", "income",
    ]

    dataset_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"

    df = pd.read_csv(
        dataset_url,
        names=columns,
        skipinitialspace=True,
        na_values=["?"],
    )

    df = df.dropna().copy()
    df = df.drop(columns=["fnlwgt", "education", "native-country"])

    # Binarize target
    df["income"] = (df["income"].str.strip() == ">50K").astype(int)

    return df, "income", "sex", "relationship"


def discretize_features(df: pd.DataFrame, n_bins: int = 5) -> pd.DataFrame:
    """
    Discretize continuous features into bins for mutual information computation.
    """
    df_disc = df.copy()

    continuous_cols = ["age", "capital-gain", "capital-loss", "hours-per-week"]

    for col in continuous_cols:
        if col in df_disc.columns:
            try:
                df_disc[col] = pd.qcut(df_disc[col], q=n_bins, labels=False, duplicates='drop')
            except Exception as e:
                print(f"Warning: Could not discretize {col}: {e}")

    return df_disc


def encode_categorical_features(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """
    Encode all categorical features to numeric values.
    """
    df_encoded = df.copy()
    label_encoders = {}

    categorical_cols = df_encoded.select_dtypes(include=['object']).columns

    for col in categorical_cols:
        le = LabelEncoder()
        df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
        label_encoders[col] = le

    return df_encoded, label_encoders


def compute_conditional_mutual_information(Y: np.ndarray, P: np.ndarray, S: np.ndarray) -> float:
    """
    Compute Conditional Mutual Information I(Y; P | S).

    Formula: I(Y; P | S) = Σ_s P(s) Σ_{y,p} P(y,p|s) log(P(y,p|s) / (P(y|s)P(p|s)))

    Args:
        Y: Target variable (1D array)
        P: Proxy variable (1D array)
        S: Sensitive attribute (1D array)

    Returns:
        Conditional mutual information value
    """
    # Ensure all inputs are 1D
    Y = np.asarray(Y).flatten()
    P = np.asarray(P).flatten()
    S = np.asarray(S).flatten()

    n = len(Y)
    cmi = 0.0

    # Get unique values
    s_values = np.unique(S)

    for s in s_values:
        # Get indices where S == s
        mask_s = (S == s)
        P_s = np.sum(mask_s) / n  # P(s)

        if P_s == 0 or np.sum(mask_s) == 0:
            continue

        # Subset data for this value of S
        Y_s = Y[mask_s]
        P_s_subset = P[mask_s]

        # Get unique values for Y and P given S
        y_values = np.unique(Y_s)
        p_values = np.unique(P_s_subset)

        n_s = np.sum(mask_s)

        # Compute conditional MI for this S value
        for y in y_values:
            for p in p_values:
                # Count co-occurrences of y and p given s
                mask_y = (Y == y)
                mask_p = (P == p)
                mask_yp_s = mask_y & mask_p & mask_s

                count_yp_s = np.sum(mask_yp_s)

                if count_yp_s == 0:
                    continue

                # P(y, p | s)
                P_yp_given_s = count_yp_s / n_s

                # P(y | s)
                count_y_s = np.sum(mask_y & mask_s)
                P_y_given_s = count_y_s / n_s if count_y_s > 0 else 0

                # P(p | s)
                count_p_s = np.sum(mask_p & mask_s)
                P_p_given_s = count_p_s / n_s if count_p_s > 0 else 0

                # Avoid log(0)
                if P_y_given_s > 0 and P_p_given_s > 0 and P_yp_given_s > 0:
                    ratio = P_yp_given_s / (P_y_given_s * P_p_given_s + 1e-10)
                    cmi += P_s * P_yp_given_s * np.log(ratio + 1e-10)

    return max(0.0, cmi)


def compute_pep(df_encoded: pd.DataFrame, sensitive_col: str,
                target_col: str) -> dict:
    """
    Compute Proxy Entanglement Penalty (PEP) for each feature.

    PEP measures how much a feature acts as a proxy for the sensitive attribute.
    Uses conditional mutual information I(Y; P | S)
    """
    Y = df_encoded[target_col].values.flatten()
    S = df_encoded[sensitive_col].values.flatten()

    pep_scores = {}

    # Compute PEP for ALL features except target and sensitive attribute
    for col in df_encoded.columns:
        if col in [target_col, sensitive_col]:
            continue

        P = df_encoded[col].values.flatten()

        # Compute I(Y; P | S)
        cmi = compute_conditional_mutual_information(Y, P, S)
        pep_scores[col] = cmi

    return pep_scores


def compute_markov_blanket_strength(df_encoded: pd.DataFrame, target_col: str,
                                   sensitive_col: str) -> dict:
    """
    Compute mutual information between each feature and the target.
    This approximates the strength of features in the Markov Blanket.
    """
    Y = df_encoded[target_col].values.flatten()

    mi_scores = {}

    # Compute MI for ALL features except target and sensitive attribute
    for col in df_encoded.columns:
        if col in [target_col, sensitive_col]:
            continue

        X = df_encoded[col].values.flatten()

        # Compute I(Y; X)
        mi = compute_mutual_information(Y, X)
        mi_scores[col] = mi

    return mi_scores


def compute_mutual_information(X: np.ndarray, Y: np.ndarray) -> float:
    """
    Compute Mutual Information I(X; Y).
    """
    # Ensure 1D
    X = np.asarray(X).flatten()
    Y = np.asarray(Y).flatten()

    n = len(X)

    x_values = np.unique(X)
    y_values = np.unique(Y)

    mi = 0.0

    for x in x_values:
        for y in y_values:
            mask_x = (X == x)
            mask_y = (Y == y)
            mask_xy = mask_x & mask_y

            count_xy = np.sum(mask_xy)
            if count_xy == 0:
                continue

            P_xy = count_xy / n
            P_x = np.sum(mask_x) / n
            P_y = np.sum(mask_y) / n

            if P_x > 0 and P_y > 0:
                mi += P_xy * np.log(P_xy / (P_x * P_y) + 1e-10)

    return max(0.0, mi)


def faircfs_feature_selection(df_encoded: pd.DataFrame, target_col: str,
                             sensitive_col: str,
                             fairness_weight: float = 0.5) -> dict:
    """
    FairCFS Algorithm: Fair Correlation-based Feature Selection.

    Balances utility (MI with target) and fairness (minimal PEP).

    Args:
        df_encoded: Encoded DataFrame
        target_col: Target column name
        sensitive_col: Sensitive attribute column name
        fairness_weight: Weight for fairness term (0=pure utility, 1=pure fairness)

    Returns:
        Dictionary with feature scores and ranking
    """

    print("\n" + "="*80)
    print("FAIRCFS FEATURE SELECTION")
    print("="*80)

    # Compute utility scores (MI with target)
    print("\n[1] Computing utility scores (MI with target)...")
    utility_scores = compute_markov_blanket_strength(df_encoded, target_col, sensitive_col)

    # Compute fairness scores (PEP)
    print("[2] Computing fairness scores (PEP)...")
    fairness_scores = compute_pep(df_encoded, sensitive_col, target_col)

    # Ensure both dictionaries have the same features
    common_features = set(utility_scores.keys()) & set(fairness_scores.keys())

    if not common_features:
        print("Error: No common features between utility and fairness scores!")
        return {}

    # Filter to common features only
    utility_scores = {k: v for k, v in utility_scores.items() if k in common_features}
    fairness_scores = {k: v for k, v in fairness_scores.items() if k in common_features}

    print(f"[3] Processing {len(common_features)} common features...")

    # Normalize scores
    utility_values = list(utility_scores.values())
    fairness_values = list(fairness_scores.values())

    max_utility = max(utility_values) if utility_values else 1.0
    max_fairness = max(fairness_values) if fairness_values else 1.0

    if max_utility > 0:
        utility_scores = {k: v / max_utility for k, v in utility_scores.items()}

    if max_fairness > 0:
        fairness_scores = {k: v / max_fairness for k, v in fairness_scores.items()}

    # Combine scores: Higher utility is good, Lower fairness penalty is good
    print("[4] Combining utility and fairness scores...")
    combined_scores = {}

    for feature in common_features:
        # Fair Score: (1 - normalized_pep) means lower PEP is better
        utility_term = utility_scores[feature]
        fairness_term = 1 - fairness_scores[feature]

        # Weighted combination
        combined_scores[feature] = (
            (1 - fairness_weight) * utility_term +
            fairness_weight * fairness_term
        )

    # Sort by combined score
    ranked_features = sorted(combined_scores.items(), key=lambda x: x[1], reverse=True)

    result = {
        'utility_scores': utility_scores,
        'fairness_scores': fairness_scores,
        'combined_scores': combined_scores,
        'ranked_features': ranked_features,
    }

    return result


def conditional_entropy(Y: np.ndarray, X: np.ndarray) -> float:
    """
    Compute conditional entropy H(Y | X).

    Args:
        Y: Target variable (1D array)
        X: Feature(s) - can be 1D or 2D

    Returns:
        Conditional entropy value
    """
    # Ensure Y is 1D
    Y = np.asarray(Y).flatten()
    X = np.asarray(X)

    # Make X 1D if it's a column vector
    if X.ndim == 2:
        if X.shape[1] == 1:
            X = X.flatten()
        else:
            # For multiple features, create tuples
            X_tuples = [tuple(row) for row in X]
            unique_X = list(set(X_tuples))
            n = len(Y)

            H_Y_given_X = 0.0

            for x_val in unique_X:
                mask = np.array([tuple(x) == x_val if X.ndim == 2 else x == x_val
                                for x in X])

                if np.sum(mask) == 0:
                    continue

                P_x = np.sum(mask) / n
                Y_given_x = Y[mask]

                if len(Y_given_x) > 0:
                    _, counts = np.unique(Y_given_x, return_counts=True)
                    probs = counts / len(Y_given_x)
                    H_Y_given_x = entropy(probs)
                    H_Y_given_X += P_x * H_Y_given_x

            return H_Y_given_X

    # For 1D X
    X = np.asarray(X).flatten()

    # Discretize X if needed
    X_unique = len(np.unique(X))
    if X_unique > 20:  # Too many bins, discretize
        X_disc = pd.qcut(X, q=5, labels=False, duplicates='drop').values
    else:
        X_disc = X

    unique_X = np.unique(X_disc)
    n = len(Y)

    H_Y_given_X = 0.0

    for x_val in unique_X:
        mask = (X_disc == x_val)

        if np.sum(mask) == 0:
            continue

        P_x = np.sum(mask) / n
        Y_given_x = Y[mask]

        if len(Y_given_x) > 0:
            _, counts = np.unique(Y_given_x, return_counts=True)
            probs = counts / len(Y_given_x)
            H_Y_given_x = entropy(probs)
            H_Y_given_X += P_x * H_Y_given_x

    return H_Y_given_X


def compute_utility_tax_fano_bound(df_encoded: pd.DataFrame, target_col: str,
                                   features_full: list[str], features_pruned: list[str]) -> float:
    """
    Compute utility tax (accuracy loss) using Fano's Inequality bound.

    ΔPe ≥ [H(Y|MB') - H(Y|MB)] / log|Y|

    where MB' = MB(Y) \ {P} (Markov Blanket after pruning proxy)
    """
    Y = df_encoded[target_col].values.flatten()

    # H(Y) - Entropy of target
    _, counts = np.unique(Y, return_counts=True)
    probs = counts / len(Y)
    H_Y = entropy(probs)

    print(f"\nH(Y) = {H_Y:.4f}")
    print(f"log|Y| = {np.log(len(np.unique(Y))):.4f}")

    # Filter features to only those that exist
    features_full_valid = [f for f in features_full if f in df_encoded.columns]
    features_pruned_valid = [f for f in features_pruned if f in df_encoded.columns]

    if not features_full_valid:
        print("Warning: No valid features for full set")
        return 0.0

    # Compute conditional entropy H(Y | MB)
    X_full = df_encoded[features_full_valid].values
    H_Y_given_MB = conditional_entropy(Y, X_full)

    # Compute conditional entropy H(Y | MB')
    if len(features_pruned_valid) > 0:
        X_pruned = df_encoded[features_pruned_valid].values
        H_Y_given_MB_prime = conditional_entropy(Y, X_pruned)
    else:
        H_Y_given_MB_prime = H_Y

    # Fano's bound
    log_Y = np.log(len(np.unique(Y)))
    fano_bound = (H_Y_given_MB_prime - H_Y_given_MB) / log_Y if log_Y > 0 else 0

    print(f"H(Y|MB) = {H_Y_given_MB:.4f}")
    print(f"H(Y|MB') = {H_Y_given_MB_prime:.4f}")
    print(f"Fano's Bound (ΔPe) = {fano_bound:.4f}")

    return fano_bound


def evaluate_feature_sets(df_encoded: pd.DataFrame, target_col: str,
                         sensitive_col: str, mb_features: list[str],
                         proxy_features: list[str]) -> dict:
    """
    Evaluate different feature sets and compute their performance metrics.
    """
    # Filter MB features to only those in df_encoded
    mb_features_valid = [f for f in mb_features if f in df_encoded.columns]
    proxy_features_valid = [f for f in proxy_features if f in df_encoded.columns]

    if not mb_features_valid:
        print("Error: No valid MB features found!")
        return {}

    X_full = df_encoded[mb_features_valid + [sensitive_col]].values
    X_mb = df_encoded[mb_features_valid].values
    X_fair = df_encoded[[f for f in mb_features_valid if f not in proxy_features_valid]].values
    Y = df_encoded[target_col].values.flatten()

    X_train_full, X_test_full, y_train, y_test = train_test_split(
        X_full, Y, test_size=0.3, random_state=42
    )
    X_train_mb, X_test_mb, _, _ = train_test_split(
        X_mb, Y, test_size=0.3, random_state=42
    )
    X_train_fair, X_test_fair, _, _ = train_test_split(
        X_fair, Y, test_size=0.3, random_state=42
    )

    results = {}

    # MB_Shield (MB + S)
    print("\n[Evaluating MB_Shield (MB + S)]")
    clf_full = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    clf_full.fit(X_train_full, y_train)
    y_pred_full = clf_full.predict(X_test_full)
    acc_full = accuracy_score(y_test, y_pred_full)
    f1_full = f1_score(y_test, y_pred_full, average='weighted', zero_division=0)
    results['MB_Shield'] = {'accuracy': acc_full, 'f1': f1_full}
    print(f"  Accuracy: {acc_full:.4f}, F1: {f1_full:.4f}")

    # MB (without S)
    print("[Evaluating MB (without S)]")
    clf_mb = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    clf_mb.fit(X_train_mb, y_train)
    y_pred_mb = clf_mb.predict(X_test_mb)
    acc_mb = accuracy_score(y_test, y_pred_mb)
    f1_mb = f1_score(y_test, y_pred_mb, average='weighted', zero_division=0)
    results['MB'] = {'accuracy': acc_mb, 'f1': f1_mb}
    print(f"  Accuracy: {acc_mb:.4f}, F1: {f1_mb:.4f}")

    # Fair_Filter (MB without P)
    print("[Evaluating Fair_Filter (MB without P)]")
    if X_train_fair.shape[1] > 0:
        clf_fair = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
        clf_fair.fit(X_train_fair, y_train)
        y_pred_fair = clf_fair.predict(X_test_fair)
        acc_fair = accuracy_score(y_test, y_pred_fair)
        f1_fair = f1_score(y_test, y_pred_fair, average='weighted', zero_division=0)
        results['Fair_Filter'] = {'accuracy': acc_fair, 'f1': f1_fair}
        print(f"  Accuracy: {acc_fair:.4f}, F1: {f1_fair:.4f}")
    else:
        results['Fair_Filter'] = {'accuracy': 0, 'f1': 0}
        print("  Not enough features after filtering proxies")

    return results


def plot_pep_analysis(pep_scores: dict, save_path: str = "pep_analysis.png"):
    """
    Visualize PEP scores for each feature.
    """
    if not pep_scores:
        print(f"Warning: No PEP scores to plot")
        return

    sorted_features = sorted(pep_scores.items(), key=lambda x: x[1], reverse=True)
    features, scores = zip(*sorted_features)

    fig, ax = plt.subplots(figsize=(12, 6))
    bars = ax.barh(features, scores, color='steelblue', alpha=0.7)

    # Color high PEP features in red
    mean_pep = np.mean(scores)
    for i, (bar, score) in enumerate(zip(bars, scores)):
        if score > mean_pep:
            bar.set_color('coral')

    ax.set_xlabel("Proxy Entanglement Penalty (PEP) - I(Y; P | S)", fontweight='bold')
    ax.set_title("Feature PEP Scores (Higher = More Proxy Entanglement)", fontweight='bold', fontsize=14)
    ax.axvline(mean_pep, color='red', linestyle='--', linewidth=2, label=f'Mean PEP: {mean_pep:.4f}')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, axis='x')

    fig.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Saved: {save_path}")
    plt.close()


def plot_feature_scores_comparison(utility_scores: dict, fairness_scores: dict,
                                   combined_scores: dict, save_path: str = "feature_scores_comparison.png"):
    """
    Compare utility, fairness, and combined scores for all features.
    """
    if not utility_scores or not fairness_scores or not combined_scores:
        print("Warning: Missing score dictionaries for plotting")
        return

    # Get common features
    common_features = set(utility_scores.keys()) & set(fairness_scores.keys()) & set(combined_scores.keys())

    if not common_features:
        print("Warning: No common features to plot")
        return

    features = list(common_features)
    utility_vals = [utility_scores[f] for f in features]
    fairness_vals = [fairness_scores[f] for f in features]
    combined_vals = [combined_scores[f] for f in features]

    # Sort by combined score
    sorted_idx = np.argsort(combined_vals)[::-1]
    features = [features[i] for i in sorted_idx]
    utility_vals = [utility_vals[i] for i in sorted_idx]
    fairness_vals = [fairness_vals[i] for i in sorted_idx]
    combined_vals = [combined_vals[i] for i in sorted_idx]

    x = np.arange(len(features))
    width = 0.25

    fig, ax = plt.subplots(figsize=(14, 6))

    ax.bar(x - width, utility_vals, width, label='Utility Score (MI)', alpha=0.8, color='steelblue')
    ax.bar(x, fairness_vals, width, label='Fairness Score (1-PEP)', alpha=0.8, color='coral')
    ax.bar(x + width, combined_vals, width, label='Combined Score', alpha=0.8, color='green')

    ax.set_xlabel("Feature", fontweight='bold', fontsize=12)
    ax.set_ylabel("Score", fontweight='bold', fontsize=12)
    ax.set_title("FairCFS Feature Scores: Utility vs Fairness (λ=0.5)", fontweight='bold', fontsize=14)
    ax.set_xticks(x)
    ax.set_xticklabels(features, rotation=45, ha='right')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim([0, 1.1])

    fig.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Saved: {save_path}")
    plt.close()


def plot_performance_comparison(results: dict, save_path: str = "performance_comparison.png"):
    """
    Compare performance across different feature sets.
    """
    if not results:
        print("Warning: No results to plot")
        return

    configs = list(results.keys())
    accuracies = [results[c]['accuracy'] for c in configs]
    f1_scores_list = [results[c]['f1'] for c in configs]

    x = np.arange(len(configs))
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 6))

    ax.bar(x - width/2, accuracies, width, label='Accuracy', alpha=0.8, color='steelblue')
    ax.bar(x + width/2, f1_scores_list, width, label='F1-Score', alpha=0.8, color='coral')

    ax.set_ylabel("Score", fontweight='bold', fontsize=12)
    ax.set_title("Model Performance Across Feature Sets", fontweight='bold', fontsize=14)
    ax.set_xticks(x)
    ax.set_xticklabels(configs, fontsize=11)
    ax.legend(fontsize=11)
    ax.set_ylim([0, 1])
    ax.grid(True, alpha=0.3, axis='y')

    # Add value labels on bars
    for bars in [ax.patches[:len(configs)], ax.patches[len(configs):]]:
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

    fig.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Saved: {save_path}")
    plt.close()


def create_summary_report(faircfs_result: dict, results: dict, fano_bound: float,
                         mb_features: list, proxy_features: list, save_path: str = "faircfs_summary_report.txt"):
    """
    Create a comprehensive text report of the analysis.
    """
    with open(save_path, 'w') as f:
        f.write("="*80 + "\n")
        f.write("FAIRCFS ANALYSIS SUMMARY REPORT\n")
        f.write("UCI Adult Dataset\n")
        f.write("="*80 + "\n\n")

        f.write("CONFIGURATION:\n")
        f.write(f"  Markov Blanket Features: {mb_features}\n")
        f.write(f"  Proxy Features: {proxy_features}\n")
        f.write(f"  Sensitive Attribute: sex\n")
        f.write(f"  Target: income\n\n")

        f.write("="*80 + "\n")
        f.write("FAIRCFS RANKING (by combined score):\n")
        f.write("="*80 + "\n")
        if faircfs_result and 'ranked_features' in faircfs_result:
            for rank, (feature, score) in enumerate(faircfs_result['ranked_features'], 1):
                utility = faircfs_result['utility_scores'].get(feature, 0)
                fairness = faircfs_result['fairness_scores'].get(feature, 0)
                f.write(f"{rank:2d}. {feature:25} | Combined: {score:.4f} | Utility: {utility:.4f} | Fairness: {fairness:.4f}\n")

        f.write("\n" + "="*80 + "\n")
        f.write("PERFORMANCE METRICS:\n")
        f.write("="*80 + "\n")
        for config, metrics in results.items():
            f.write(f"\n{config}:\n")
            f.write(f"  Accuracy: {metrics['accuracy']:.4f}\n")
            f.write(f"  F1-Score: {metrics['f1']:.4f}\n")

        f.write("\n" + "="*80 + "\n")
        f.write("FANO'S INEQUALITY BOUND (Utility Tax):\n")
        f.write("="*80 + "\n")
        f.write(f"ΔPe >= {fano_bound:.4f}\n")
        f.write(f"\nInterpretation: Removing proxy features introduces at least\n")
        f.write(f"a {fano_bound:.4f} probability of error increase (theoretical lower bound)\n")

        f.write("\n" + "="*80 + "\n")
        f.write("KEY INSIGHTS:\n")
        f.write("="*80 + "\n")

        if faircfs_result and 'ranked_features' in faircfs_result:
            top_fair = [f for f, s in faircfs_result['ranked_features'][:3]]
            f.write(f"\n1. Top 3 Fairness-Utility Balanced Features:\n")
            for i, feat in enumerate(top_fair, 1):
                f.write(f"   {i}. {feat}\n")

        if results:
            max_acc_config = max(results.items(), key=lambda x: x[1]['accuracy'])
            min_acc_config = min(results.items(), key=lambda x: x[1]['accuracy'])
            f.write(f"\n2. Accuracy Comparison:\n")
            f.write(f"   Best: {max_acc_config[0]} ({max_acc_config[1]['accuracy']:.4f})\n")
            f.write(f"   Worst: {min_acc_config[0]} ({min_acc_config[1]['accuracy']:.4f})\n")
            f.write(f"   Difference: {max_acc_config[1]['accuracy'] - min_acc_config[1]['accuracy']:.4f}\n")

        f.write("\n" + "="*80 + "\n")

    print(f"✓ Saved summary report: {save_path}")


if __name__ == "__main__":
    print("="*80)
    print("FAIRCFS ALGORITHM WITH PEP COMPUTATION")
    print("UCI Adult Dataset")
    print("="*80)

    # Load and preprocess data
    print("\n[1/6] Loading UCI Adult Dataset...")
    df, target_col, sensitive_col, proxy_col = load_and_preprocess_adult()
    print(f"  ✓ Dataset shape: {df.shape}")
    print(f"  ✓ Columns: {df.columns.tolist()}")

    # Discretize continuous features
    print("\n[2/6] Discretizing continuous features...")
    df_disc = discretize_features(df, n_bins=5)
    print(f"  ✓ Discretization complete")

    # Encode categorical features
    print("[3/6] Encoding categorical features...")
    df_encoded, encoders = encode_categorical_features(df_disc)
    print(f"  ✓ Encoded dataset shape: {df_encoded.shape}")
    print(f"  ✓ All columns encoded: {list(df_encoded.columns)}")

    # Define Markov Blanket features
    mb_features = [
        "age",
        "education-num",
        "marital-status",
        "occupation",
        "hours-per-week",
    ]

    # Define proxy features (features correlated with sensitive attribute)
    proxy_features = ["marital-status", "occupation"]

    print(f"\nConfiguration:")
    print(f"  Markov Blanket (MB): {mb_features}")
    print(f"  Proxy Features (P): {proxy_features}")
    print(f"  Sensitive Attribute (S): {sensitive_col}")
    print(f"  Target (Y): {target_col}")

    # Run FairCFS
    print("\n[4/6] Running FairCFS algorithm...")
    faircfs_result = faircfs_feature_selection(
        df_encoded,
        target_col=target_col,
        sensitive_col=sensitive_col,
        fairness_weight=0.5  # Balance utility and fairness
    )

    if not faircfs_result:
        print("Error: FairCFS failed!")
        exit(1)

    # Display results
    print("\n" + "="*80)
    print("FAIRCFS RESULTS")
    print("="*80)

    print("\n[Utility Scores - I(Y; Feature)]")
    for feature, score in sorted(faircfs_result['utility_scores'].items(),
                                 key=lambda x: x[1], reverse=True):
        print(f"  {feature:25} : {score:.4f}")

    print("\n[Fairness Scores - (1 - PEP)]")
    for feature, score in sorted(faircfs_result['fairness_scores'].items(),
                                 key=lambda x: x[1], reverse=True):
        print(f"  {feature:25} : {score:.4f}")

    print("\n[Combined Scores - λ=0.5]")
    for feature, score in faircfs_result['ranked_features']:
        print(f"  {feature:25} : {score:.4f}")

    # Compute Fano's bound
    print("\n[5/6] Computing Fano's Inequality Bound...")
    print("="*80)
    print("FANO'S INEQUALITY BOUND (Utility Tax)")
    print("="*80)

    mb_features_valid = [f for f in mb_features if f in df_encoded.columns]
    proxy_features_valid = [f for f in proxy_features if f in df_encoded.columns]
    pruned_features = [f for f in mb_features_valid if f not in proxy_features_valid]

    fano_bound = compute_utility_tax_fano_bound(
        df_encoded,
        target_col=target_col,
        features_full=mb_features_valid + [sensitive_col],
        features_pruned=pruned_features
    )

    # Evaluate different feature sets
    print("\n" + "="*80)
    print("PERFORMANCE EVALUATION")
    print("="*80)

    results = evaluate_feature_sets(
        df_encoded,
        target_col=target_col,
        sensitive_col=sensitive_col,
        mb_features=mb_features,
        proxy_features=proxy_features
    )

    print("\n" + "="*80)
    print("SUMMARY TABLE")
    print("="*80)

    summary_df = pd.DataFrame([
        {
            'Feature Set': 'MB_Shield (MB+S)',
            'Accuracy': results['MB_Shield']['accuracy'],
            'F1-Score': results['MB_Shield']['f1'],
            'Description': 'Markov Blanket + Sensitive Attribute'
        },
        {
            'Feature Set': 'MB',
            'Accuracy': results['MB']['accuracy'],
            'F1-Score': results['MB']['f1'],
            'Description': 'Markov Blanket only'
        },
        {
            'Feature Set': 'Fair_Filter',
            'Accuracy': results['Fair_Filter']['accuracy'],
            'F1-Score': results['Fair_Filter']['f1'],
            'Description': 'MB without Proxy Features'
        },
    ])

    print(summary_df.to_string(index=False))

    # Generate visualizations
    print("\n[6/6] Generating visualizations...")
    print("="*80)

    plot_pep_analysis(faircfs_result['fairness_scores'])
    plot_feature_scores_comparison(
        faircfs_result['utility_scores'],
        faircfs_result['fairness_scores'],
        faircfs_result['combined_scores']
    )
    plot_performance_comparison(results)

    # Create summary report
    create_summary_report(faircfs_result, results, fano_bound, mb_features, proxy_features)

    print("\n" + "="*80)
    print("✓ ANALYSIS COMPLETE!")
    print("="*80)
    print("\nGenerated Files:")
    print("  • pep_analysis.png")
    print("  • feature_scores_comparison.png")
    print("  • performance_comparison.png")
    print("  • faircfs_summary_report.txt")
    print("="*80)

FAIRCFS ALGORITHM WITH PEP COMPUTATION
UCI Adult Dataset

[1/6] Loading UCI Adult Dataset...
  ✓ Dataset shape: (30162, 12)
  ✓ Columns: ['age', 'workclass', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'income']

[2/6] Discretizing continuous features...
  ✓ Discretization complete
[3/6] Encoding categorical features...
  ✓ Encoded dataset shape: (30162, 12)
  ✓ All columns encoded: ['age', 'workclass', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'income']

Configuration:
  Markov Blanket (MB): ['age', 'education-num', 'marital-status', 'occupation', 'hours-per-week']
  Proxy Features (P): ['marital-status', 'occupation']
  Sensitive Attribute (S): sex
  Target (Y): income

[4/6] Running FairCFS algorithm...

FAIRCFS FEATURE SELECTION

[1] Computing utility scores (MI with target)...
[2] Computing fairness scores (PE